# Figure 9.25: Ground-Truth Noise and Performance Metrics

Noisy labels can make a model look worse even when its error relative to the latent clean trend stays stable.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


def rng_from_seed(seed):
    return np.random.default_rng(int(seed))


def linspace(a, b, n):
    return np.linspace(a, b, int(n))


def base_curve(x):
    return 1.0 + 0.7 * x - 0.35 * x**2 + 0.06 * x**3


def poly_design(x, degree):
    return np.vstack([x**k for k in range(degree + 1)]).T


def ridge_fit(x, y, degree, lam):
    X = poly_design(x, degree)
    I = np.eye(degree + 1)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def poly_predict(theta, x):
    X = poly_design(np.asarray(x), len(theta) - 1)
    return X @ theta


def line_fit(x, y, lam=0.0):
    X = np.column_stack([np.ones_like(x), x])
    I = np.eye(2)
    I[0, 0] = 0.0
    return np.linalg.pinv(X.T @ X + lam * I) @ X.T @ y


def line_predict(theta, x):
    x = np.asarray(x)
    return theta[0] + theta[1] * x


def poly_data(seed=0, n=40, obs_noise=0.3, gt_noise=0.0):
    rng = rng_from_seed(seed)
    x = np.linspace(-3, 3, n)
    y_true = base_curve(x)
    y_star = y_true + gt_noise * rng.normal(size=n)
    y = y_star + obs_noise * rng.normal(size=n)
    return x, y, y_true, y_star

def draw(gt_noise=0.25, obs_noise=0.20, degree=5, lam=2.0):
    x_tr, y_tr, y_true_tr, _ = poly_data(seed=31, n=50, obs_noise=obs_noise, gt_noise=gt_noise)
    x_te, y_te, y_true_te, _ = poly_data(seed=37, n=80, obs_noise=obs_noise, gt_noise=gt_noise)
    th = ridge_fit(x_tr, y_tr, degree, lam)
    xg = np.linspace(-3, 3, 300)
    fig, axs = plt.subplots(1, 2, figsize=(13.0, 4.4))
    axs[0].scatter(x_tr, y_tr, color="#334155", s=18, label="observed labels")
    axs[0].plot(xg, base_curve(xg), "--", color="#2563eb", lw=2.5, label="latent clean trend")
    axs[0].plot(xg, poly_predict(th, xg), color="#dc2626", lw=2.6, label="ridge fit")
    axs[0].set_title("Latent trend vs noisy labels")
    axs[0].grid(alpha=0.2)
    axs[0].legend()
    pred_te = poly_predict(th, x_te)
    noisy_mse = np.mean((pred_te - y_te) ** 2)
    clean_mse = np.mean((pred_te - y_true_te) ** 2)
    label_noise = np.mean((y_te - y_true_te) ** 2)
    axs[1].bar(["noisy labels", "clean truth", "noise floor"], [noisy_mse, clean_mse, label_noise], color=["#475569", "#2563eb", "#d97706"])
    axs[1].set_ylabel("MSE")
    axs[1].set_title("Which target are we evaluating against?")
    axs[1].grid(alpha=0.2)
    plt.show()
    print(f"Measured against noisy labels = {noisy_mse:.3f}")
    print(f"Measured against clean truth = {clean_mse:.3f}")

widgets.interact(
    draw,
    gt_noise=widgets.FloatSlider(min=0.0, max=0.8, step=0.05, value=0.25, continuous_update=False),
    obs_noise=widgets.FloatSlider(min=0.0, max=0.6, step=0.05, value=0.20, continuous_update=False),
    degree=widgets.IntSlider(min=1, max=10, step=1, value=5, continuous_update=False),
    lam=widgets.FloatSlider(min=0.0, max=20.0, step=0.5, value=2.0, continuous_update=False),
)
